In [2]:
import os
import pandas as pd
PATH = r"C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\2.source_code\Step5_Geo_RF_trial\Food_Crisis_Cluster\results"

In [4]:
# set pattern

pattern = 'y_pred_test_gp_fs1_*_*.csv'
import glob
all_filenames = [i for i in glob.glob(os.path.join(PATH, pattern))]
# combine all files in the list
combined_csv_scope_1 = pd.concat([pd.read_csv(f) for f in all_filenames ])

In [5]:
# scope 2 and 3
pattern = 'y_pred_test_gp_fs2_*_*.csv'
all_filenames = [i for i in glob.glob(os.path.join(PATH, pattern))]
combined_csv_scope_2 = pd.concat([pd.read_csv(f) for f in all_filenames ])
pattern = 'y_pred_test_gp_fs3_*_*.csv'
all_filenames = [i for i in glob.glob(os.path.join(PATH, pattern))]
combined_csv_scope_3 = pd.concat([pd.read_csv(f) for f in all_filenames ])

In [6]:
# for each scope calculate month level precision, recall, f1, pred label is fews_ipc_crisis_pred, true label is fews_ipc_crisis_true
from sklearn.metrics import precision_score, recall_score, f1_score
def calculate_metrics(df):
    metrics = {}
    for month in range(1, 13):
        df_month = df[df['month'] == month]
        y_true = df_month['fews_ipc_crisis_true']
        y_pred = df_month['fews_ipc_crisis_pred']
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        metrics[month] = {'precision': precision, 'recall': recall, 'f1': f1}
    return pd.DataFrame.from_dict(metrics, orient='index')

metrics_scope_1 = calculate_metrics(combined_csv_scope_1)
metrics_scope_2 = calculate_metrics(combined_csv_scope_2)
metrics_scope_3 = calculate_metrics(combined_csv_scope_3)

In [7]:
# calculate seasonality, first group months into 3 seasons, season 1: Jan-Apr, season 2: May-Aug, season 3: Sep-Dec, then calculate precision, recall, f1 for each season
def calculate_seasonality_metrics(df):
    seasonality_metrics = {}
    seasons = {
        'season_1': [1, 2, 3, 4],
        'season_2': [5, 6, 7, 8],
        'season_3': [9, 10, 11, 12]
    }
    for season, months in seasons.items():
        df_season = df[df['month'].isin(months)]
        y_true = df_season['fews_ipc_crisis_true']
        y_pred = df_season['fews_ipc_crisis_pred']
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        seasonality_metrics[season] = {'precision': precision, 'recall': recall, 'f1': f1}
    return pd.DataFrame.from_dict(seasonality_metrics, orient='index')

seasonality_metrics_scope_1 = calculate_seasonality_metrics(combined_csv_scope_1)
seasonality_metrics_scope_2 = calculate_seasonality_metrics(combined_csv_scope_2)
seasonality_metrics_scope_3 = calculate_seasonality_metrics(combined_csv_scope_3)

In [8]:
# calculate after 2021 monthly and seasonality metrics
def filter_after_2021(df):
    return df[df['year'] > 2021]

metrics_scope_1_after_2021 = calculate_metrics(filter_after_2021(combined_csv_scope_1))
metrics_scope_2_after_2021 = calculate_metrics(filter_after_2021(combined_csv_scope_2))
metrics_scope_3_after_2021 = calculate_metrics(filter_after_2021(combined_csv_scope_3))
seasonality_metrics_scope_1_after_2021 = calculate_seasonality_metrics(filter_after_2021(combined_csv_scope_1))
seasonality_metrics_scope_2_after_2021 = calculate_seasonality_metrics(filter_after_2021(combined_csv_scope_2))
seasonality_metrics_scope_3_after_2021 = calculate_seasonality_metrics(filter_after_2021(combined_csv_scope_3))

In [9]:
# every dataframe keep only 3 digits
def round_metrics(df):
    return df.round(3)
metrics_scope_1 = round_metrics(metrics_scope_1)
metrics_scope_2 = round_metrics(metrics_scope_2)
metrics_scope_3 = round_metrics(metrics_scope_3)
seasonality_metrics_scope_1 = round_metrics(seasonality_metrics_scope_1)
seasonality_metrics_scope_2 = round_metrics(seasonality_metrics_scope_2)
seasonality_metrics_scope_3 = round_metrics(seasonality_metrics_scope_3)
metrics_scope_1_after_2021 = round_metrics(metrics_scope_1_after_2021)
metrics_scope_2_after_2021 = round_metrics(metrics_scope_2_after_2021)
metrics_scope_3_after_2021 = round_metrics(metrics_scope_3_after_2021)
seasonality_metrics_scope_1_after_2021 = round_metrics(seasonality_metrics_scope_1_after_2021)
seasonality_metrics_scope_2_after_2021 = round_metrics(seasonality_metrics_scope_2_after_2021)
seasonality_metrics_scope_3_after_2021 = round_metrics(seasonality_metrics_scope_3_after_2021)

In [10]:
# add month to metrics_scope dataframes
metrics_scope_1.insert(0, 'month', range(1, 13))
metrics_scope_2.insert(0, 'month', range(1, 13))
metrics_scope_3.insert(0, 'month', range(1, 13))
seasonality_metrics_scope_1.insert(0, 'season', ['season_1', 'season_2', 'season_3'])
seasonality_metrics_scope_2.insert(0, 'season', ['season_1', 'season_2', 'season_3'])
seasonality_metrics_scope_3.insert(0, 'season', ['season_1', 'season_2', 'season_3'])
metrics_scope_1_after_2021.insert(0, 'month', range(1, 13))
metrics_scope_2_after_2021.insert(0, 'month', range(1, 13))
metrics_scope_3_after_2021.insert(0, 'month', range(1, 13))
seasonality_metrics_scope_1_after_2021.insert(0, 'season', ['season_1', 'season_2', 'season_3'])
seasonality_metrics_scope_2_after_2021.insert(0, 'season', ['season_1', 'season_2', 'season_3'])
seasonality_metrics_scope_3_after_2021.insert(0, 'season', ['season_1', 'season_2', 'season_3'])

In [11]:
# export to csv
metrics_scope_1.to_csv(os.path.join(PATH, 'metrics_scope_1.csv'), index=False)
metrics_scope_2.to_csv(os.path.join(PATH, 'metrics_scope_2.csv'), index=False)
metrics_scope_3.to_csv(os.path.join(PATH, 'metrics_scope_3.csv'), index=False)
seasonality_metrics_scope_1.to_csv(os.path.join(PATH, 'seasonality_metrics_scope_1.csv'), index=False)
seasonality_metrics_scope_2.to_csv(os.path.join(PATH, 'seasonality_metrics_scope_2.csv'), index=False)
seasonality_metrics_scope_3.to_csv(os.path.join(PATH, 'seasonality_metrics_scope_3.csv'), index=False)
metrics_scope_1_after_2021.to_csv(os.path.join(PATH, 'metrics_scope_1_after_2021.csv'), index=False)
metrics_scope_2_after_2021.to_csv(os.path.join(PATH, 'metrics_scope_2_after_2021.csv'), index=False)
metrics_scope_3_after_2021.to_csv(os.path.join(PATH, 'metrics_scope_3_after_2021.csv'), index=False)
seasonality_metrics_scope_1_after_2021.to_csv(os.path.join(PATH, 'seasonality_metrics_scope_1_after_2021.csv'), index=False)
seasonality_metrics_scope_2_after_2021.to_csv(os.path.join(PATH, 'seasonality_metrics_scope_2_after_2021.csv'), index=False)
seasonality_metrics_scope_3_after_2021.to_csv(os.path.join(PATH, 'seasonality_metrics_scope_3_after_2021.csv'), index=False)

In [12]:
# summarize yearly metrics for each scope
def calculate_yearly_metrics(df):
    yearly_metrics = {}
    years = df['year'].unique()
    for year in years:
        df_year = df[df['year'] == year]
        y_true = df_year['fews_ipc_crisis_true']
        y_pred = df_year['fews_ipc_crisis_pred']
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        yearly_metrics[year] = {'precision': precision, 'recall': recall, 'f1': f1}
    return pd.DataFrame.from_dict(yearly_metrics, orient='index')

yearly_metrics_scope_1 = calculate_yearly_metrics(combined_csv_scope_1)
yearly_metrics_scope_2 = calculate_yearly_metrics(combined_csv_scope_2)
yearly_metrics_scope_3 = calculate_yearly_metrics(combined_csv_scope_3)

# add year to yearly_metrics dataframes
yearly_metrics_scope_1.insert(0, 'year', yearly_metrics_scope_1.index)
yearly_metrics_scope_2.insert(0, 'year', yearly_metrics_scope_2.index)
yearly_metrics_scope_3.insert(0, 'year', yearly_metrics_scope_3.index)
# export yearly metrics to csv
yearly_metrics_scope_1.to_csv(os.path.join(PATH, 'yearly_metrics_scope_1.csv'), index=False)
yearly_metrics_scope_2.to_csv(os.path.join(PATH, 'yearly_metrics_scope_2.csv'), index=False)
yearly_metrics_scope_3.to_csv(os.path.join(PATH, 'yearly_metrics_scope_3.csv'), index=False)

In [13]:
DATA_PATH = r"C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\FEWSNET_forecast_unadjusted_bm.csv"


#import data
data = pd.read_csv(DATA_PATH)


C:\Users\swl00\AppData\Local\Temp\ipykernel_7976\1515880029.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(DATA_PATH)


In [16]:
# filter only fews_ipc_crisis is not missing
data_filtered = data[~data['fews_ipc_crisis'].isna()]

In [18]:
# value counts of country
data_filtered['ADMIN0'].value_counts()

ADMIN0
Ethiopia                            51900
Kenya                               32588
Afghanistan                         23414
Guatemala                           18003
Sudan                               17689
Uganda                              16147
Nigeria                             14484
Mozambique                          11883
Zimbabwe                            10385
Somalia                              9702
Niger                                9282
Haiti                                8874
Mali                                 5865
Democratic Republic of the Congo     5520
Chad                                 4743
Yemen                                4344
Madagascar                           4158
Malawi                               4061
South Sudan                          3633
Cameroon                             1440
Burundi                               675
Burkina Faso                          650
Name: count, dtype: int64

In [19]:
# filter Malawi
data_malawi = data_filtered[data_filtered['ADMIN0'] == 'Malawi']

# see value counts of fews_ipc_crisis in Malawi
data_malawi['fews_ipc_crisis'].value_counts()

fews_ipc_crisis
0.0    3714
1.0     347
Name: count, dtype: int64

In [21]:
# pasrse date for data_malawi
data_malawi['date'] = pd.to_datetime(data_malawi['date'])

C:\Users\swl00\AppData\Local\Temp\ipykernel_7976\1514417892.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_malawi['date'] = pd.to_datetime(data_malawi['date'])


In [22]:
#for data_malawi, filter date >= '2021-01-01'
data_malawi_after_2021 = data_malawi[data_malawi['date'] >= '2021-01-01']

In [23]:
data_malawi_after_2021['fews_ipc_crisis'].value_counts()

fews_ipc_crisis
0.0    705
1.0    169
Name: count, dtype: int64